# DATA 715 | Module 11: Document Store and Key-Value Store

### Self-contained classroom demonstration

This notebook uses **SQLite in memory** to simulate two storage patterns:

- **Document store:** JSON documents encrypted and stored as binary values.
- **Key-value store:** A document ID maps to the encryption key used for that document.

It requires **no PostgreSQL server, database credentials, files, or network connection**. All data is recreated each time the notebook is run from the top. The `cryptography` Python package is needed for Fernet encryption; if it is not installed, run the optional installation cell below.

**Learning goals:** understand JSON serialization, key-value lookups, encryption and decryption, and security tradeoffs.

## 1. Optional dependency installation

Run the following cell **only if** `cryptography` is missing from your Python environment. Package installation may require internet access the first time. SQLite and JSON are included with Python.

In [ ]:
# Optional, only if the next section raises ModuleNotFoundError:
# %pip install cryptography

## 2. Create local storage

SQLite's `:memory:` option creates a temporary relational database inside the notebook process. It is used here strictly as a **local storage engine** to demonstrate document and key-value patterns, not as a claim that SQLite is a native document database.

The `documents` table holds encrypted JSON blobs. The `key_store` table demonstrates mapping a document ID to a key.

In [ ]:
import json
import sqlite3
from cryptography.fernet import Fernet, InvalidToken

# Fresh, in-memory database. No host, account, password, or saved DB file.
conn = sqlite3.connect(":memory:")
conn.execute("PRAGMA foreign_keys = ON")

conn.executescript("""
CREATE TABLE key_store (
    doc_id TEXT PRIMARY KEY,
    key BLOB NOT NULL
);
CREATE TABLE documents (
    doc_id TEXT PRIMARY KEY,
    content BLOB NOT NULL
);
""")
print("Local in-memory document and key-value stores created.")

## 3. Store and retrieve encrypted JSON documents

Each document is assigned a unique ID and its own Fernet key. JSON is serialized to UTF-8 bytes before encryption. SQLite uses parameterized queries (`?`) for safe value substitution.

**Important:** Keeping encryption keys and encrypted data in the same database is suitable for illustrating lookup logic but **not a secure production key-management design**. Real applications should use separate, access-controlled key management.

In [ ]:
def generate_key():
    return Fernet.generate_key()


def save_key(doc_id, key):
    conn.execute("INSERT INTO key_store (doc_id, key) VALUES (?, ?)", (doc_id, key))
    conn.commit()


def get_key(doc_id):
    row = conn.execute("SELECT key FROM key_store WHERE doc_id = ?", (doc_id,)).fetchone()
    return row[0] if row else None


def add_document(doc_id, content):
    """Encrypt and store a JSON-serializable object and its lookup key."""
    if not isinstance(doc_id, str) or not doc_id:
        raise ValueError("doc_id must be a nonempty string")
    key = generate_key()
    encrypted_content = Fernet(key).encrypt(json.dumps(content).encode("utf-8"))
    # A single transaction prevents a key without its corresponding document.
    with conn:
        conn.execute("INSERT INTO key_store (doc_id, key) VALUES (?, ?)", (doc_id, key))
        conn.execute("INSERT INTO documents (doc_id, content) VALUES (?, ?)",
                     (doc_id, encrypted_content))


def get_document(doc_id):
    """Retrieve, decrypt, and reconstruct the original Python object."""
    key = get_key(doc_id)
    row = conn.execute("SELECT content FROM documents WHERE doc_id = ?", (doc_id,)).fetchone()
    if key is None or row is None:
        return None
    try:
        return json.loads(Fernet(key).decrypt(row[0]).decode("utf-8"))
    except InvalidToken as exc:
        raise ValueError(f"Could not decrypt document {doc_id!r}") from exc

## 4. Example: insert, inspect, and retrieve

The encrypted value is opaque binary ciphertext. Decrypting it produces the original nested JSON document.

In [ ]:
doc_id = "doc_1"
document_content = {
    "title": "Sensitive Document",
    "body": "This is a secure document.",
    "tags": ["DATA715", "document-store"],
    "metadata": {"course": "DATA 715", "module": 11}
}

add_document(doc_id, document_content)
print("Original document:")
print(json.dumps(document_content, indent=2))

ciphertext = conn.execute("SELECT content FROM documents WHERE doc_id = ?", (doc_id,)).fetchone()[0]
print("\nEncrypted content (first 70 bytes):", ciphertext[:70])
print("\nDecrypted document:")
print(json.dumps(get_document(doc_id), indent=2))
assert get_document(doc_id) == document_content

## 5. Compare the two storage patterns

A **key-value lookup** uses `doc_id` to return a single value (the key). A **document lookup** uses the same ID to retrieve a structured JSON document. Notice that our example uses relational tables underneath both abstractions.

In [ ]:
print("KEY-VALUE LOOKUP")
print("Document ID:", doc_id)
print("Stored key (first 16 bytes):", get_key(doc_id)[:16])

print("\nDOCUMENT LOOKUP")
print("Title:", get_document(doc_id)["title"])
print("Tags:", get_document(doc_id)["tags"])

print("\nMISSING DOCUMENT")
print(get_document("not_found"))

## 6. Decrypt and materialize a readable copy (for demonstration only)

The original notebook also wrote decrypted content into a separate table. Here, the same operation is performed **in memory**. This deliberately creates plaintext copies and is **not** a recommended production practice. There is no reason to duplicate encryption keys in a decrypted table.

In [ ]:
conn.execute("""
CREATE TABLE documents_decrypt (
    doc_id TEXT PRIMARY KEY,
    content TEXT NOT NULL
)
""")


def decrypt_and_store_document(doc_id):
    content = get_document(doc_id)
    if content is None:
        print(f"No document or key found for {doc_id}")
        return None
    with conn:
        conn.execute(
            "INSERT OR REPLACE INTO documents_decrypt (doc_id, content) VALUES (?, ?)",
            (doc_id, json.dumps(content))
        )
    print(f"Decrypted JSON temporarily materialized for {doc_id} (classroom use only).")
    return content


decrypt_and_store_document("doc_1")
row = conn.execute("SELECT content FROM documents_decrypt WHERE doc_id = ?", ("doc_1",)).fetchone()
print(json.dumps(json.loads(row[0]), indent=2))

## 7. Practice: add another document

Try a document with **nested objects** and **arrays**, then retrieve it by ID. What happens if you add a duplicate `doc_id`? Why does a document store differ from a simple string key-value store?

In [ ]:
student_document = {
    "student_id": "s123",
    "courses": [
        {"code": "DATA715", "grade": "A"},
        {"code": "DATA710", "grade": "B"}
    ],
    "active": True
}
add_document("student_1", student_document)
print(json.dumps(get_document("student_1"), indent=2))
assert get_document("student_1")["courses"][0]["code"] == "DATA715"

## 8. Discussion

1. What are the advantages and limitations of storing JSON as one document rather than normalizing it into relational tables?
2. When would you **embed** course records inside the student document? When would you **reference** them instead?
3. What are the risks of storing encryption keys alongside the encrypted data?
4. How might **massive arrays** become a document design anti-pattern if a student accumulates millions of records?
5. Would you use SQLite, a document database such as MongoDB, or a key-value service such as DynamoDB for this application? Defend your choice.

**Cleanup:** the database disappears when the kernel stops. You can also run `conn.close()` after you finish.